# Enrichment Pipeline

VulnFlow enriches CVE data with multiple sources. Some enrichments run by default,
others are opt-in. This notebook shows what's available and how to control it.

| Default (on) | Opt-in (off) |
|---|---|
| CVSS flatten, EPSS, KEV, CWE, CPE | CAPEC, STRIDE |

In [ ]:
from vulnflow.helpers import CacheBuilder

# Build once with all enrichments enabled (defaults + opt-ins)
ds = CacheBuilder(years=[2025]).build(capec=True, stride=True)
print(f"{len(ds):,} CVEs  |  {len(ds.columns)} columns")
print(f"Columns: {ds.columns}")

## Default Enrichments

These are applied automatically by `CacheBuilder.build()`:

| Enrichment | Source | Key Columns |
|---|---|---|
| CVSS flatten | NVD scores | `cvss_base_score`, `cvss_base_severity`, `cvss_attack_vector` |
| EPSS | FIRST EPSS API | `epss_probability`, `epss_percentile` |
| KEV | CISA catalog | `is_kev`, `kev_date_added`, `kev_ransomware` |
| CWE | MITRE CWE DB | `cwe_id`, `cwe_name`, `cwe_category_name` |
| CPE | NVD CPE data | `cpe_vendor`, `cpe_product`, `cpe_version` |

In [2]:
df = ds.to_pandas()

# CVSS
scored = df["cvss_base_score"].notna().sum()
print(f"CVSS scores: {scored:,} / {len(df):,}")
print(f"  Severity: {df['cvss_base_severity'].value_counts().to_dict()}")

# EPSS
epss = df["epss_probability"].notna().sum()
print(f"\nEPSS scores: {epss:,} / {len(df):,}")
print(f"  Mean probability: {df['epss_probability'].mean():.4f}")

# KEV
kev_count = (df["is_kev"] == True).sum()
print(f"\nKEV entries: {kev_count:,}")

# CWE
cwe = df["cwe_id"].notna().sum()
print(f"\nCWE assigned: {cwe:,} / {len(df):,}")
print(f"  Top 3: {df['cwe_id'].value_counts().head(3).to_dict()}")

# CPE
cpe = df["cpe_vendor"].notna().sum()
print(f"\nCPE assigned: {cpe:,} / {len(df):,}")
print(f"  Top 3 vendors: {df['cpe_vendor'].value_counts().head(3).to_dict()}")

CVSS scores: 39,984 / 40,804
  Severity: {'MEDIUM': 18538, 'HIGH': 14521, 'CRITICAL': 5930, 'LOW': 956, 'NONE': 39}

EPSS scores: 40,733 / 40,804
  Mean probability: 0.0057

KEV entries: 161

CWE assigned: 38,978 / 40,804
  Top 3: {'CWE-79': 7274, 'CWE-74': 2559, 'CWE-862': 2105}

CPE assigned: 23,999 / 40,804
  Top 3 vendors: {'linux': 1796, 'microsoft': 1240, 'adobe': 818}


## Opt-in: CAPEC Attack Patterns

CAPEC maps CWEs to known attack patterns. Requires CWE enrichment (on by default).

In [ ]:
import json
import matplotlib.pyplot as plt
from collections import Counter

df_nested = ds.to_pandas(include_nested=True)

# Count CAPEC patterns across all CVEs
capec_counter: Counter[str] = Counter()
for raw in df_nested["capecs"].dropna():
    entries = json.loads(raw) if isinstance(raw, str) else raw
    for entry in entries:
        if isinstance(entry, dict) and "name" in entry:
            capec_counter[entry["name"]] += 1

print(f"{sum(capec_counter.values()):,} CAPEC mappings across {len(capec_counter)} patterns\n")

top_capec = capec_counter.most_common(10)
names = [c[0][:50] for c in top_capec]
values = [c[1] for c in top_capec]

fig, ax = plt.subplots(figsize=(14, 6))
ax.barh(names[::-1], values[::-1], color="#1f77b4", edgecolor="white")
ax.set_title("Top 10 CAPEC Attack Patterns", fontsize=16)
ax.set_xlabel("CVE Count", fontsize=12)
ax.grid(True, linestyle="--", alpha=0.5, axis="x")
plt.tight_layout()
plt.show()

## Opt-in: STRIDE Threat Categories

STRIDE infers threat categories (Spoofing, Tampering, Repudiation, Information
Disclosure, Denial of Service, Elevation of Privilege) from CAPEC, CWE, and CVSS data.

In [ ]:
# STRIDE categories were already computed in the initial build
stride_counter: Counter[str] = Counter()
for raw in df_nested["stride_threats"].dropna():
    entries = json.loads(raw) if isinstance(raw, str) else raw
    for entry in entries:
        if isinstance(entry, dict) and "category" in entry:
            stride_counter[entry["category"]] += 1

print(f"STRIDE distribution: {dict(stride_counter)}\n")

labels = list(stride_counter.keys())
sizes = list(stride_counter.values())
colors = ["#e74c3c", "#f39c12", "#f1c40f", "#3498db", "#2ecc71", "#9b59b6"][:len(labels)]

fig, ax = plt.subplots(figsize=(8, 6))
ax.pie(sizes, labels=labels, colors=colors, autopct="%1.0f%%", startangle=90,
       textprops={"fontsize": 11})
ax.set_title("STRIDE Threat Distribution", fontsize=16)
plt.tight_layout()
plt.show()

## Build with All Enrichments

One-liner equivalent of what we did above — pass `capec=True, stride=True` to `build()`:

```python
ds = CacheBuilder(years=[2025]).build(capec=True, stride=True)
```

In [ ]:
# Already built with all enrichments — just verify
print(f"{len(ds):,} CVEs  |  {len(ds.columns)} columns")
print(f"Columns: {ds.columns}")

## Manual Transform Pipeline

For full control, pass explicit transforms to `ds.transform()`.

In [ ]:
from vulnflow.transforms.enrichers.epss import EPSSEnricher
from vulnflow.transforms.enrichers.kev import KEVEnricher
from vulnflow.transforms.enrichers.cwe import CWEEnricher
from vulnflow.transforms.flatten.cvss import CVSSFlattenTransform
from vulnflow.transforms.flatten.epss import EPSSFlattenTransform
from vulnflow.transforms.flatten.kev import KEVFlattenTransform
from vulnflow.transforms.flatten.cwe import CWEFlattenTransform
from vulnflow.transforms.flatten.cpe import CPEFlattenTransform

# Build without enrichments (second build — needed to demo manual workflow)
ds_raw = CacheBuilder(years=[2025]).build(
    cvss=False, epss=False, kev=False, cwe=False, cpe=False,
)
print(f"Raw columns ({len(ds_raw.columns)}): {ds_raw.columns}")

ds_raw.transform(transforms=[
    EPSSEnricher(),
    KEVEnricher(),
    CWEEnricher(),
    CVSSFlattenTransform(),
    EPSSFlattenTransform(),
    KEVFlattenTransform(),
    CWEFlattenTransform(),
    CPEFlattenTransform(),
])
print(f"After transforms ({len(ds_raw.columns)}): {ds_raw.columns}")

## Controlling Enrichment via build() kwargs

Skip specific enrichments by passing `False`.

In [ ]:
# build(cvss=False) skips CVSS flattening — same principle as ds_raw above
# Example: CacheBuilder(years=[2025]).build(cvss=False)
#
# You can disable any enrichment individually:
#   build(epss=False)   — skip EPSS scores
#   build(kev=False)    — skip KEV catalog
#   build(cwe=False)    — skip CWE weakness names
#   build(cpe=False)    — skip CPE vendor/product extraction

# Verify: ds (all enrichments) has cvss_base_score, ds_raw (before transforms) did not
print(f"Full build has cvss_base_score: {'cvss_base_score' in ds.columns}")
print(f"Full build has epss_probability: {'epss_probability' in ds.columns}")
print(f"Full build has is_kev: {'is_kev' in ds.columns}")